# D4BL Training Pipeline — Qwen 3.5-4B

Run the headless training script on Colab A100. Trains 3 LoRA adapters (query parser, explainer, evaluator) with domain adaptation, then exports to GGUF.

**Runtime:** Change to **A100** GPU via Runtime > Change runtime type.

**Time estimate:** ~30-50 minutes on A100.

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install dependencies

In [ ]:
import subprocess, sys

# Unsloth (includes transformers, peft, trl, etc.)
subprocess.run([sys.executable, "-m", "pip", "install", "unsloth"], check=True)

# HuggingFace datasets
subprocess.run([sys.executable, "-m", "pip", "install", "datasets"], check=True)

## 3. Clone repo and set up

In [ ]:
import os
os.chdir("/content")

# Clone if not already present
if not os.path.exists("/content/d4bl_ai_agent"):
    !git clone https://github.com/William-Hill/d4bl_ai_agent.git

os.chdir("/content/d4bl_ai_agent")
!git pull origin main
!pwd

## 4. (Optional) HuggingFace token for gated models

In [ ]:
# Uncomment and set if needed for gated model access
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

## 5. Run training

This runs all 5 phases: domain adaptation, 3 task adapters, and GGUF export. Progress streams in real-time. If interrupted, rerun this cell — it resumes from checkpoints.

In [ ]:
!python scripts/training/train.py \
    --data-dir scripts/training_data/final \
    --output-dir /content/d4bl_training

## 6. Review results

In [ ]:
# Training report
with open("/content/d4bl_training/training_report.md") as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

In [ ]:
# List GGUF exports
!ls -lh /content/d4bl_training/gguf/*/

## 7. Download GGUFs

Download the GGUF files to use locally with Ollama.

In [ ]:
from google.colab import files
import glob

for gguf in glob.glob("/content/d4bl_training/gguf/**/*.gguf", recursive=True):
    print(f"Downloading {gguf}...")
    files.download(gguf)